<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/model_evaluation/numerical_answer_evaluation_using_llm-as-judge_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Numerical Answer Evaluation using LLM-as-Judge

This notebook demonstrates how to use a Large Language Model (LLM) as a judge to evaluate numerical accuracy in model-generated answers.

## Approach Overview

### Why LLM-as-Judge for Numerical Evaluation?

Traditional rule-based number extraction can miss context. An LLM can:
1. **Understand semantic context** - Know that "revenue" in one text matches "sales" in another
2. **Handle ambiguity** - Resolve which numbers should be compared
3. **Explain decisions** - Provide reasoning for accept/reject decisions
4. **Handle edge cases** - Deal with different formats, units, and representations

### Pipeline:
1. **LLM extracts numbers** with semantic labels from both texts
2. **LLM matches number pairs** based on semantic meaning
3. **Statistical comparison** of matched pairs
4. **LLM makes final judgment** with explanation

## Use Cases:
- Fact-checking numerical claims in generated text
- Evaluating math problem solutions
- Quality control for RAG systems with numerical data
- Comparing financial/scientific data in LLM outputs


## 1. Installation and Setup


In [ ]:
# Install required packages
!pip install -q openai pandas numpy matplotlib seaborn plotly python-dotenv pydantic


In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field, asdict
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# OpenAI
from openai import OpenAI

# Pydantic for structured outputs
from pydantic import BaseModel, Field

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Environment
from dotenv import load_dotenv

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

# Load environment variables
load_dotenv()

print("✅ All imports successful!")


## 2. Configure LLM Client

Set up the OpenAI client. You can use OpenAI's API or any compatible endpoint (Azure OpenAI, local models via Ollama, etc.).


In [ ]:
# Configure OpenAI client
# Option 1: Use environment variable OPENAI_API_KEY
# Option 2: Set directly (not recommended for production)

# For Colab, you can set it like this:
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Initialize the client
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    # base_url="http://localhost:11434/v1"  # Uncomment for Ollama
)

# Model to use for evaluation
LLM_MODEL = "gpt-4o-mini"  # or "gpt-4o", "gpt-4-turbo", etc.

print(f"✅ OpenAI client configured")
print(f"📌 Using model: {LLM_MODEL}")


## 3. Define Structured Output Models

Using Pydantic models to get structured JSON responses from the LLM.


In [ ]:
# Pydantic models for structured LLM outputs

class ExtractedNumber(BaseModel):
    """A number extracted from text by the LLM."""
    value: float = Field(description="The numerical value")
    original_text: str = Field(description="The original text representation of the number")
    semantic_label: str = Field(description="What this number represents (e.g., 'revenue', 'population', 'percentage growth')")
    unit: Optional[str] = Field(default=None, description="Unit of measurement if applicable")
    context: str = Field(description="Brief context around the number")


class NumberExtractionResult(BaseModel):
    """Result of extracting numbers from a text."""
    numbers: List[ExtractedNumber] = Field(description="List of extracted numbers with context")
    extraction_notes: str = Field(description="Any notes about the extraction process")


class NumberPair(BaseModel):
    """A matched pair of numbers from model answer and ground truth."""
    model_number: ExtractedNumber = Field(description="Number from the model answer")
    truth_number: ExtractedNumber = Field(description="Corresponding number from ground truth")
    match_confidence: str = Field(description="Confidence level: 'high', 'medium', 'low'")
    match_reasoning: str = Field(description="Why these numbers were matched")


class NumberMatchingResult(BaseModel):
    """Result of matching numbers between model answer and ground truth."""
    matched_pairs: List[NumberPair] = Field(description="List of matched number pairs")
    unmatched_model_numbers: List[ExtractedNumber] = Field(description="Numbers in model answer not matched")
    unmatched_truth_numbers: List[ExtractedNumber] = Field(description="Numbers in ground truth not matched")
    matching_notes: str = Field(description="Notes about the matching process")


class NumericalJudgment(BaseModel):
    """Final judgment from the LLM judge."""
    decision: str = Field(description="Overall decision: 'accept', 'marginal', or 'reject'")
    overall_score: float = Field(description="Score from 0.0 to 1.0")
    reasoning: str = Field(description="Detailed reasoning for the decision")
    pair_assessments: List[Dict[str, Any]] = Field(description="Assessment for each matched pair")
    recommendations: str = Field(description="Recommendations for improvement if rejected")


print("✅ Pydantic models defined")


## 4. LLM-Based Number Extraction

The LLM extracts numbers from text with semantic understanding of what each number represents.


In [ ]:
class LLMNumberExtractor:
    """
    Uses an LLM to extract numbers from text with semantic understanding.
    """
    
    EXTRACTION_PROMPT = """You are an expert at extracting numerical information from text.

Given the following text, extract ALL numbers mentioned along with:
1. The actual numerical value (convert to standard notation, e.g., "$2.5 billion" -> 2500000000)
2. The original text representation
3. A semantic label describing what the number represents
4. The unit of measurement (if any)
5. Brief context around the number

Text to analyze:
\"\"\"
{text}
\"\"\"

Extract all numbers, including:
- Explicit numbers (42, 3.14, 1,000)
- Percentages (15%, 0.5%)
- Currency amounts ($100, €50)
- Large numbers with words (2.5 billion, 10 million)
- Fractions (1/2, 3/4)
- Dates/years only if they represent quantities (not calendar dates)

Be thorough and extract every numerical value mentioned."""

    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model
    
    def extract(self, text: str) -> NumberExtractionResult:
        """Extract numbers from text using LLM."""
        try:
            response = self.client.beta.chat.completions.parse(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a precise number extraction assistant. Always return valid JSON."},
                    {"role": "user", "content": self.EXTRACTION_PROMPT.format(text=text)}
                ],
                response_format=NumberExtractionResult,
                temperature=0.0
            )
            return response.choices[0].message.parsed
        except Exception as e:
            print(f"Error in extraction: {e}")
            # Fallback to manual parsing
            return self._fallback_extraction(text)
    
    def _fallback_extraction(self, text: str) -> NumberExtractionResult:
        """Fallback extraction using regex if LLM fails."""
        numbers = []
        # Simple regex patterns
        patterns = [
            (r'\$[\d,]+\.?\d*\s*(?:billion|million|trillion)?', 'currency'),
            (r'[\d,]+\.?\d*\s*%', 'percentage'),
            (r'[\d,]+\.?\d*\s*(?:billion|million|trillion)', 'large_number'),
            (r'[\d,]+\.?\d+', 'decimal'),
            (r'[\d,]+', 'integer'),
        ]
        
        found_positions = set()
        for pattern, num_type in patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                if match.start() not in found_positions:
                    found_positions.add(match.start())
                    try:
                        value_str = re.sub(r'[,$%]', '', match.group())
                        value_str = re.sub(r'\s*(billion|trillion|million)\s*', '', value_str, flags=re.IGNORECASE)
                        value = float(value_str)
                        
                        multiplier = 1
                        if 'trillion' in match.group().lower():
                            multiplier = 1e12
                        elif 'billion' in match.group().lower():
                            multiplier = 1e9
                        elif 'million' in match.group().lower():
                            multiplier = 1e6
                        
                        numbers.append(ExtractedNumber(
                            value=value * multiplier,
                            original_text=match.group(),
                            semantic_label=num_type,
                            unit=None,
                            context=text[max(0, match.start()-30):min(len(text), match.end()+30)]
                        ))
                    except ValueError:
                        continue
        
        return NumberExtractionResult(
            numbers=numbers,
            extraction_notes="Fallback regex extraction used"
        )


# Initialize extractor
extractor = LLMNumberExtractor(client, LLM_MODEL)

# Test extraction
test_text = """
Apple reported Q3 2023 revenue of $81.8 billion, representing a 1.4% year-over-year decline. 
The gross margin was 44.5%, while operating income reached $23.1 billion. 
Earnings per share came in at $1.26, with a total of 15.8 billion shares outstanding.
"""

print("📊 Testing LLM Number Extraction")
print("="*70)
print(f"\nInput text:\n{test_text}")

extraction_result = extractor.extract(test_text)

print(f"\n✅ Extracted {len(extraction_result.numbers)} numbers:")
for num in extraction_result.numbers:
    print(f"\n  • {num.semantic_label}: {num.value:,.4g}")
    print(f"    Original: '{num.original_text}'")
    print(f"    Unit: {num.unit}")
print(f"\n📝 Notes: {extraction_result.extraction_notes}")


## 5. LLM-Based Number Matching

The LLM matches numbers from the model answer to corresponding numbers in the ground truth based on semantic meaning.


## 6. Statistical Comparison Module

Calculate statistical metrics for matched number pairs.


In [ ]:
@dataclass
class PairStatistics:
    """Statistical comparison of a number pair."""
    model_value: float
    truth_value: float
    absolute_error: float
    relative_error: float
    percentage_error: float
    order_of_magnitude_diff: float
    is_exact_match: bool
    semantic_label: str
    match_confidence: str


class StatisticalComparator:
    """
    Computes statistical metrics for matched number pairs.
    """
    
    def __init__(
        self,
        relative_tolerance: float = 0.10,
        absolute_tolerance: float = 0.01,
        order_magnitude_tolerance: float = 1.0
    ):
        self.relative_tolerance = relative_tolerance
        self.absolute_tolerance = absolute_tolerance
        self.order_magnitude_tolerance = order_magnitude_tolerance
    
    def compare_pair(self, pair: NumberPair) -> PairStatistics:
        """Compute statistics for a single pair."""
        model_val = pair.model_number.value
        truth_val = pair.truth_number.value
        
        # Absolute error
        abs_error = abs(model_val - truth_val)
        
        # Relative error
        if truth_val != 0:
            rel_error = abs_error / abs(truth_val)
        else:
            rel_error = float('inf') if model_val != 0 else 0.0
        
        # Percentage error
        pct_error = rel_error * 100
        
        # Order of magnitude difference
        if model_val > 0 and truth_val > 0:
            magnitude_diff = abs(np.log10(model_val) - np.log10(truth_val))
        elif model_val == 0 and truth_val == 0:
            magnitude_diff = 0.0
        else:
            magnitude_diff = float('inf')
        
        # Check if exact match (within floating point tolerance)
        is_exact = abs_error < 1e-9 or rel_error < 1e-9
        
        return PairStatistics(
            model_value=model_val,
            truth_value=truth_val,
            absolute_error=abs_error,
            relative_error=rel_error,
            percentage_error=pct_error,
            order_of_magnitude_diff=magnitude_diff,
            is_exact_match=is_exact,
            semantic_label=pair.model_number.semantic_label,
            match_confidence=pair.match_confidence
        )
    
    def is_within_tolerance(self, stats: PairStatistics) -> bool:
        """Check if comparison is within acceptable tolerance."""
        # For small numbers, use absolute tolerance
        if abs(stats.truth_value) < 1.0:
            return stats.absolute_error <= self.absolute_tolerance
        
        # For larger numbers, use relative tolerance
        return (
            stats.relative_error <= self.relative_tolerance and
            stats.order_of_magnitude_diff <= self.order_magnitude_tolerance
        )
    
    def compare_all(self, matching_result: NumberMatchingResult) -> List[PairStatistics]:
        """Compare all matched pairs."""
        return [self.compare_pair(pair) for pair in matching_result.matched_pairs]
    
    def get_summary(self, stats_list: List[PairStatistics]) -> Dict[str, Any]:
        """Get summary statistics."""
        if not stats_list:
            return {
                'total_pairs': 0,
                'within_tolerance': 0,
                'exact_matches': 0,
                'avg_relative_error': 0,
                'max_relative_error': 0,
                'avg_absolute_error': 0
            }
        
        within_tol = sum(1 for s in stats_list if self.is_within_tolerance(s))
        exact = sum(1 for s in stats_list if s.is_exact_match)
        rel_errors = [s.relative_error for s in stats_list if s.relative_error != float('inf')]
        abs_errors = [s.absolute_error for s in stats_list]
        
        return {
            'total_pairs': len(stats_list),
            'within_tolerance': within_tol,
            'exact_matches': exact,
            'accuracy_ratio': within_tol / len(stats_list),
            'avg_relative_error': np.mean(rel_errors) if rel_errors else float('inf'),
            'max_relative_error': max(rel_errors) if rel_errors else float('inf'),
            'avg_absolute_error': np.mean(abs_errors)
        }


# Initialize comparator
comparator = StatisticalComparator(
    relative_tolerance=0.10,  # 10%
    absolute_tolerance=0.01,
    order_magnitude_tolerance=1.0
)

print("✅ StatisticalComparator initialized")


## 7. LLM Judge

The LLM makes a final judgment considering both statistical metrics and semantic context.


In [ ]:
class LLMJudge:
    """
    LLM-based judge that makes final decisions on numerical accuracy.
    """
    
    JUDGMENT_PROMPT = """You are an expert judge evaluating the numerical accuracy of a model's answer compared to ground truth.

QUESTION: {question}

MODEL ANSWER: {model_answer}

GROUND TRUTH: {ground_truth}

MATCHED NUMBER PAIRS AND STATISTICS:
{pair_stats}

SUMMARY STATISTICS:
- Total matched pairs: {total_pairs}
- Pairs within tolerance ({tolerance}%): {within_tolerance}
- Exact matches: {exact_matches}
- Average relative error: {avg_rel_error:.2%}
- Maximum relative error: {max_rel_error:.2%}

UNMATCHED NUMBERS:
- Model answer has {unmatched_model} unmatched numbers
- Ground truth has {unmatched_truth} unmatched numbers

Based on this analysis, provide:
1. An overall decision: 'accept' (accurate enough), 'marginal' (borderline), or 'reject' (too inaccurate)
2. A score from 0.0 to 1.0
3. Detailed reasoning explaining your decision
4. Assessment for each matched pair
5. Recommendations for improvement if the answer is rejected

Consider:
- Are the errors acceptable for the type of data (financial, scientific, general)?
- Do small errors compound to create a misleading picture?
- Are critical numbers (main figures) accurate even if secondary numbers aren't?
- Is the overall narrative/conclusion correct despite numerical differences?"""

    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model
    
    def _format_pair_stats(
        self, 
        matching_result: NumberMatchingResult, 
        stats_list: List[PairStatistics]
    ) -> str:
        """Format pair statistics for the prompt."""
        lines = []
        for pair, stats in zip(matching_result.matched_pairs, stats_list):
            lines.append(f"• {stats.semantic_label}:")
            lines.append(f"  Model: {stats.model_value:,.4g} | Truth: {stats.truth_value:,.4g}")
            lines.append(f"  Relative Error: {stats.relative_error:.2%}")
            lines.append(f"  Match Confidence: {stats.match_confidence}")
            lines.append(f"  Reason: {pair.match_reasoning}")
            lines.append("")
        return "\n".join(lines)
    
    def judge(
        self,
        question: str,
        model_answer: str,
        ground_truth: str,
        matching_result: NumberMatchingResult,
        stats_list: List[PairStatistics],
        summary: Dict[str, Any],
        tolerance: float = 10.0
    ) -> NumericalJudgment:
        """Make a final judgment using the LLM."""
        try:
            response = self.client.beta.chat.completions.parse(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an expert numerical accuracy judge. Provide fair, detailed assessments."},
                    {"role": "user", "content": self.JUDGMENT_PROMPT.format(
                        question=question,
                        model_answer=model_answer,
                        ground_truth=ground_truth,
                        pair_stats=self._format_pair_stats(matching_result, stats_list),
                        total_pairs=summary.get('total_pairs', 0),
                        within_tolerance=summary.get('within_tolerance', 0),
                        exact_matches=summary.get('exact_matches', 0),
                        avg_rel_error=summary.get('avg_relative_error', 0),
                        max_rel_error=summary.get('max_relative_error', 0),
                        tolerance=tolerance,
                        unmatched_model=len(matching_result.unmatched_model_numbers),
                        unmatched_truth=len(matching_result.unmatched_truth_numbers)
                    )}
                ],
                response_format=NumericalJudgment,
                temperature=0.0
            )
            return response.choices[0].message.parsed
        except Exception as e:
            print(f"Error in judgment: {e}")
            return self._fallback_judgment(summary)
    
    def _fallback_judgment(self, summary: Dict[str, Any]) -> NumericalJudgment:
        """Fallback judgment based on statistics only."""
        accuracy_ratio = summary.get('accuracy_ratio', 0)
        
        if accuracy_ratio >= 0.7:
            decision = "accept"
            score = 0.8 + 0.2 * accuracy_ratio
        elif accuracy_ratio >= 0.5:
            decision = "marginal"
            score = 0.4 + 0.4 * accuracy_ratio
        else:
            decision = "reject"
            score = accuracy_ratio * 0.5
        
        return NumericalJudgment(
            decision=decision,
            overall_score=score,
            reasoning=f"Based on statistical analysis: {summary.get('within_tolerance', 0)}/{summary.get('total_pairs', 0)} pairs within tolerance",
            pair_assessments=[],
            recommendations="Review numbers with high error rates"
        )


# Initialize judge
judge = LLMJudge(client, LLM_MODEL)

print("✅ LLMJudge initialized")


## 8. Complete Evaluation Pipeline

Combining all components into a unified pipeline.


In [ ]:
@dataclass
class FullEvaluationResult:
    """Complete evaluation result with all details."""
    question: str
    model_answer: str
    ground_truth: str
    
    # Extraction results
    model_numbers: NumberExtractionResult
    truth_numbers: NumberExtractionResult
    
    # Matching results
    matching_result: NumberMatchingResult
    
    # Statistical comparison
    pair_statistics: List[PairStatistics]
    summary_stats: Dict[str, Any]
    
    # Final judgment
    judgment: NumericalJudgment


class LLMNumericalEvaluator:
    """
    Complete pipeline for evaluating numerical accuracy using LLM-as-Judge.
    
    Pipeline:
    1. Extract numbers from model answer and ground truth using LLM
    2. Match corresponding number pairs using LLM
    3. Compute statistical metrics for each pair
    4. Make final judgment using LLM
    """
    
    def __init__(
        self,
        client: OpenAI,
        model: str = "gpt-4o-mini",
        relative_tolerance: float = 0.10,
        absolute_tolerance: float = 0.01
    ):
        self.client = client
        self.model = model
        
        # Initialize components
        self.extractor = LLMNumberExtractor(client, model)
        self.matcher = LLMNumberMatcher(client, model)
        self.comparator = StatisticalComparator(
            relative_tolerance=relative_tolerance,
            absolute_tolerance=absolute_tolerance
        )
        self.judge = LLMJudge(client, model)
        
        self.relative_tolerance = relative_tolerance
    
    def evaluate(
        self, 
        question: str, 
        model_answer: str, 
        ground_truth: str,
        verbose: bool = True
    ) -> FullEvaluationResult:
        """
        Run the complete evaluation pipeline.
        """
        if verbose:
            print("🔄 Step 1: Extracting numbers from model answer...")
        model_numbers = self.extractor.extract(model_answer)
        
        if verbose:
            print(f"   Found {len(model_numbers.numbers)} numbers")
            print("🔄 Step 2: Extracting numbers from ground truth...")
        truth_numbers = self.extractor.extract(ground_truth)
        
        if verbose:
            print(f"   Found {len(truth_numbers.numbers)} numbers")
            print("🔄 Step 3: Matching number pairs...")
        matching_result = self.matcher.match(model_numbers.numbers, truth_numbers.numbers)
        
        if verbose:
            print(f"   Matched {len(matching_result.matched_pairs)} pairs")
            print("🔄 Step 4: Computing statistical metrics...")
        pair_stats = self.comparator.compare_all(matching_result)
        summary = self.comparator.get_summary(pair_stats)
        
        if verbose:
            print(f"   {summary.get('within_tolerance', 0)}/{summary.get('total_pairs', 0)} within tolerance")
            print("🔄 Step 5: Making final judgment...")
        judgment = self.judge.judge(
            question=question,
            model_answer=model_answer,
            ground_truth=ground_truth,
            matching_result=matching_result,
            stats_list=pair_stats,
            summary=summary,
            tolerance=self.relative_tolerance * 100
        )
        
        if verbose:
            print(f"✅ Evaluation complete: {judgment.decision.upper()}")
        
        return FullEvaluationResult(
            question=question,
            model_answer=model_answer,
            ground_truth=ground_truth,
            model_numbers=model_numbers,
            truth_numbers=truth_numbers,
            matching_result=matching_result,
            pair_statistics=pair_stats,
            summary_stats=summary,
            judgment=judgment
        )


# Initialize the evaluator
evaluator = LLMNumericalEvaluator(
    client=client,
    model=LLM_MODEL,
    relative_tolerance=0.10
)

print("✅ LLMNumericalEvaluator pipeline initialized")


## 9. Sample Evaluation Examples

Let's test the evaluator with various scenarios.


In [ ]:
# Sample evaluation examples
examples = [
    {
        "id": 1,
        "name": "Financial Report - Accurate",
        "question": "What were Apple's Q3 2023 financial results?",
        "model_answer": """Apple reported revenue of $81.8 billion for Q3 2023, representing a 
        1.4% year-over-year decline. The gross margin was 44.5%, while operating income 
        reached $23.1 billion. Earnings per share came in at $1.26.""",
        "ground_truth": """Apple's Q3 2023 revenue was $81.8 billion, down 1.4% from the 
        prior year. Gross margin stood at 44.5%, operating income was $23.1 billion, 
        and EPS was $1.26."""
    },
    {
        "id": 2,
        "name": "Scientific Data - Minor Error",
        "question": "What is the speed of light and Earth's distance from the Sun?",
        "model_answer": """The speed of light in a vacuum is approximately 299,792 kilometers 
        per second. Earth is located at an average distance of 150 million kilometers from 
        the Sun, also known as 1 AU (astronomical unit).""",
        "ground_truth": """Light travels at exactly 299,792.458 kilometers per second in a 
        vacuum. The Earth orbits the Sun at a mean distance of 149.6 million kilometers, 
        which defines 1 astronomical unit (AU)."""
    },
    {
        "id": 3,
        "name": "Population Statistics - Significant Error",
        "question": "What is the population of Tokyo and New York City?",
        "model_answer": """Tokyo has a population of approximately 14.5 million people in 
        the city proper, making it one of the largest cities in the world. New York City 
        has around 9 million residents.""",
        "ground_truth": """The city proper of Tokyo has about 13.96 million inhabitants. 
        New York City's population is approximately 8.34 million people."""
    },
    {
        "id": 4,
        "name": "Economic Data - Order of Magnitude Error",
        "question": "What is the US GDP and national debt?",
        "model_answer": """The US GDP is approximately 25 trillion dollars. The national 
        debt stands at around 340 billion dollars as of 2023.""",
        "ground_truth": """The United States has a GDP of about $25.5 trillion. The 
        national debt has reached approximately $34 trillion."""
    },
    {
        "id": 5,
        "name": "Math Problem - Calculation Error",
        "question": "If a company has $50 million revenue and 25% profit margin, what is the profit?",
        "model_answer": """With $50 million in revenue and a 25% profit margin, the 
        company's profit would be $15 million.""",
        "ground_truth": """Given $50 million revenue and 25% profit margin, the profit 
        is $12.5 million (50 × 0.25 = 12.5)."""
    },
]

print(f"📝 Created {len(examples)} evaluation examples")


In [ ]:
# Run evaluation on all examples
results = []

print("🔍 Running LLM-as-Judge Evaluations")
print("="*80)

for ex in examples:
    print(f"\n{'='*80}")
    print(f"📊 Example {ex['id']}: {ex['name']}")
    print("="*80)
    
    result = evaluator.evaluate(
        question=ex["question"],
        model_answer=ex["model_answer"],
        ground_truth=ex["ground_truth"],
        verbose=True
    )
    
    results.append({
        "example": ex,
        "result": result
    })
    
    # Print judgment summary
    decision_emoji = {
        "accept": "✅",
        "marginal": "⚠️",
        "reject": "❌"
    }
    
    print(f"\n📋 JUDGMENT:")
    print(f"   Decision: {decision_emoji.get(result.judgment.decision, '❓')} {result.judgment.decision.upper()}")
    print(f"   Score: {result.judgment.overall_score:.2f}")
    print(f"\n   Reasoning: {result.judgment.reasoning[:200]}...")
    print(f"\n   Recommendations: {result.judgment.recommendations[:150]}...")


## 10. Results Visualization


In [ ]:
# Create summary DataFrame
summary_data = []
for r in results:
    ex = r["example"]
    res = r["result"]
    summary_data.append({
        "Example": ex["name"],
        "Decision": res.judgment.decision,
        "Score": res.judgment.overall_score,
        "Model Numbers": len(res.model_numbers.numbers),
        "Truth Numbers": len(res.truth_numbers.numbers),
        "Matched Pairs": len(res.matching_result.matched_pairs),
        "Within Tolerance": res.summary_stats.get('within_tolerance', 0),
        "Avg Relative Error": res.summary_stats.get('avg_relative_error', 0)
    })

summary_df = pd.DataFrame(summary_data)

print("📊 Evaluation Summary")
print("="*100)
display(summary_df)


In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Decision colors
decision_colors = {
    'accept': '#2ecc71',
    'marginal': '#f39c12', 
    'reject': '#e74c3c'
}

# Plot 1: Overall Scores
ax1 = axes[0, 0]
bars = ax1.barh(summary_df['Example'], summary_df['Score'])
for bar, decision in zip(bars, summary_df['Decision']):
    bar.set_color(decision_colors.get(decision, 'gray'))
ax1.axvline(x=0.7, color='green', linestyle='--', alpha=0.7, label='Accept threshold')
ax1.axvline(x=0.5, color='orange', linestyle='--', alpha=0.7, label='Marginal threshold')
ax1.set_xlabel('LLM Judge Score')
ax1.set_title('LLM-as-Judge Scores by Example')
ax1.legend()
ax1.set_xlim(0, 1)

# Plot 2: Decision Distribution
ax2 = axes[0, 1]
decision_counts = summary_df['Decision'].value_counts()
colors_pie = [decision_colors.get(d, 'gray') for d in decision_counts.index]
wedges, texts, autotexts = ax2.pie(
    decision_counts.values, 
    labels=decision_counts.index, 
    autopct='%1.1f%%',
    colors=colors_pie,
    explode=[0.05] * len(decision_counts)
)
ax2.set_title('Decision Distribution')

# Plot 3: Numbers Matched
ax3 = axes[1, 0]
x = range(len(summary_df))
width = 0.35
ax3.bar([i - width/2 for i in x], summary_df['Truth Numbers'], width, label='Ground Truth Numbers', color='#34495e')
ax3.bar([i + width/2 for i in x], summary_df['Matched Pairs'], width, label='Matched Pairs', color='#2ecc71')
ax3.set_xticks(x)
ax3.set_xticklabels([ex['name'][:15] + '...' for ex in examples], rotation=45, ha='right')
ax3.set_ylabel('Count')
ax3.set_title('Number Matching Coverage')
ax3.legend()

# Plot 4: Error Distribution
ax4 = axes[1, 1]
errors = summary_df['Avg Relative Error'].replace([np.inf, -np.inf], np.nan).fillna(0)
bars = ax4.bar(range(len(summary_df)), errors * 100)
for bar, decision in zip(bars, summary_df['Decision']):
    bar.set_color(decision_colors.get(decision, 'gray'))
ax4.axhline(y=10, color='green', linestyle='--', alpha=0.7, label='10% tolerance')
ax4.set_xticks(range(len(summary_df)))
ax4.set_xticklabels([ex['name'][:15] + '...' for ex in examples], rotation=45, ha='right')
ax4.set_ylabel('Average Relative Error (%)')
ax4.set_title('Average Error by Example')
ax4.legend()

plt.tight_layout()
plt.show()


## 11. Detailed Judgment Analysis

Let's examine the LLM's reasoning for each evaluation.


In [ ]:
def display_detailed_judgment(result: FullEvaluationResult, example_name: str):
    """Display detailed judgment information."""
    decision_emoji = {
        "accept": "✅",
        "marginal": "⚠️",
        "reject": "❌"
    }
    
    print(f"\n{'='*80}")
    print(f"🔎 Detailed Judgment: {example_name}")
    print(f"{'='*80}")
    
    judgment = result.judgment
    
    print(f"\n📊 DECISION: {decision_emoji.get(judgment.decision, '❓')} {judgment.decision.upper()}")
    print(f"📈 SCORE: {judgment.overall_score:.2f}/1.00")
    
    print(f"\n📝 REASONING:")
    print(f"   {judgment.reasoning}")
    
    print(f"\n📋 MATCHED NUMBER PAIRS:")
    for i, (pair, stats) in enumerate(zip(result.matching_result.matched_pairs, result.pair_statistics), 1):
        within = "✓" if comparator.is_within_tolerance(stats) else "✗"
        print(f"\n   {i}. {stats.semantic_label}")
        print(f"      Model: {stats.model_value:,.4g} | Truth: {stats.truth_value:,.4g}")
        print(f"      Error: {stats.relative_error:.2%} [{within}]")
        print(f"      Match confidence: {pair.match_confidence}")
    
    if result.matching_result.unmatched_model_numbers:
        print(f"\n⚠️ UNMATCHED MODEL NUMBERS:")
        for num in result.matching_result.unmatched_model_numbers:
            print(f"   • {num.semantic_label}: {num.value:,.4g}")
    
    if result.matching_result.unmatched_truth_numbers:
        print(f"\n⚠️ UNMATCHED GROUND TRUTH NUMBERS:")
        for num in result.matching_result.unmatched_truth_numbers:
            print(f"   • {num.semantic_label}: {num.value:,.4g}")
    
    print(f"\n💡 RECOMMENDATIONS:")
    print(f"   {judgment.recommendations}")

# Display detailed judgments for select examples
display_detailed_judgment(results[0]["result"], results[0]["example"]["name"])
display_detailed_judgment(results[3]["result"], results[3]["example"]["name"])


## 12. Best Practices and Recommendations


In [ ]:
best_practices = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║              BEST PRACTICES FOR LLM-AS-JUDGE NUMERICAL EVALUATION               ║
╚══════════════════════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 1. MODEL SELECTION                                                              │
├─────────────────────────────────────────────────────────────────────────────────┤
│ • GPT-4o/GPT-4-turbo: Best accuracy, higher cost                               │
│ • GPT-4o-mini: Good balance of cost and accuracy (recommended for most cases)   │
│ • Claude 3.5 Sonnet: Excellent alternative with strong reasoning               │
│ • Local models (Llama 3, Mixtral): Cost-effective for high volume              │
│                                                                                 │
│ TIP: Use structured outputs (response_format) for consistent parsing           │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 2. TOLERANCE SETTINGS BY DOMAIN                                                 │
├─────────────────────────────────────────────────────────────────────────────────┤
│ FINANCIAL DATA:                                                                 │
│   relative_tolerance = 0.01-0.05 (1-5% error acceptable)                       │
│   Critical: Exact figures for regulatory reporting                             │
│                                                                                 │
│ SCIENTIFIC DATA:                                                                │
│   relative_tolerance = 0.005-0.02 (0.5-2% error for precision)                 │
│   Consider significant figures and measurement uncertainty                      │
│                                                                                 │
│ GENERAL KNOWLEDGE/TRIVIA:                                                       │
│   relative_tolerance = 0.10-0.20 (10-20% error may be acceptable)              │
│   Order of magnitude matters more than precision                                │
│                                                                                 │
│ MATHEMATICAL CALCULATIONS:                                                      │
│   relative_tolerance = 0.001-0.01 (very strict for correctness)                │
│   Use absolute_tolerance for results near zero                                  │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 3. ADVANTAGES OF LLM-AS-JUDGE                                                   │
├─────────────────────────────────────────────────────────────────────────────────┤
│ ✅ Semantic Understanding: Matches "revenue" with "sales", "profit" with       │
│    "earnings", understands context                                              │
│                                                                                 │
│ ✅ Handles Ambiguity: Resolves which numbers should be compared when           │
│    multiple numbers exist                                                       │
│                                                                                 │
│ ✅ Explains Decisions: Provides reasoning that can be audited and understood   │
│                                                                                 │
│ ✅ Contextual Judgment: Considers whether errors matter for the overall        │
│    narrative or conclusion                                                      │
│                                                                                 │
│ ✅ Handles Edge Cases: Deals with different formats, units, representations    │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 4. LIMITATIONS & CONSIDERATIONS                                                 │
├─────────────────────────────────────────────────────────────────────────────────┤
│ ⚠️ Non-deterministic: Same input may produce different outputs                 │
│    Solution: Use temperature=0, run multiple evaluations if needed             │
│                                                                                 │
│ ⚠️ API Costs: Each evaluation requires multiple API calls                      │
│    Solution: Batch processing, caching, use smaller models for screening       │
│                                                                                 │
│ ⚠️ Latency: Slower than rule-based approaches                                  │
│    Solution: Async processing, combine with fast pre-filters                   │
│                                                                                 │
│ ⚠️ Token Limits: Long texts may exceed context windows                         │
│    Solution: Chunking, summarization, or use models with larger context        │
└─────────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────────┐
│ 5. COMPARISON: LLM-AS-JUDGE vs OTHER APPROACHES                                │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│ | Aspect              | LLM-as-Judge | Rule-Based | Sentence Transformers |   │
│ |---------------------|--------------|------------|----------------------|     │
│ | Semantic Match      | Excellent    | Poor       | Good                 |     │
│ | Explainability      | Excellent    | Limited    | Limited              |     │
│ | Speed               | Slow         | Very Fast  | Fast                 |     │
│ | Cost                | High         | Free       | Free (local)         |     │
│ | Determinism         | Low          | Perfect    | High                 |     │
│ | Edge Case Handling  | Excellent    | Poor       | Medium               |     │
│                                                                                 │
│ RECOMMENDATION: Use LLM-as-Judge for high-stakes evaluations where            │
│ explainability and nuanced understanding matter. Use faster approaches for     │
│ high-volume screening.                                                          │
└─────────────────────────────────────────────────────────────────────────────────┘
"""

print(best_practices)


## 13. Summary

This notebook demonstrated **LLM-as-Judge** for numerical answer evaluation:

### Key Components:

1. **LLMNumberExtractor**: Uses LLM to extract numbers with semantic labels and context
2. **LLMNumberMatcher**: Uses LLM to match number pairs based on semantic meaning
3. **StatisticalComparator**: Computes error metrics for matched pairs
4. **LLMJudge**: Makes final accept/marginal/reject decision with reasoning

### Pipeline Flow:

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Question + Model Answer + Ground Truth                                 │
│                          │                                              │
│                          ▼                                              │
│  ┌──────────────────────────────────────┐                              │
│  │  LLM Number Extraction               │                              │
│  │  • Extract numbers with context      │                              │
│  │  • Add semantic labels               │                              │
│  └──────────────────────────────────────┘                              │
│                          │                                              │
│                          ▼                                              │
│  ┌──────────────────────────────────────┐                              │
│  │  LLM Number Matching                 │                              │
│  │  • Match by semantic meaning         │                              │
│  │  • Handle synonyms and context       │                              │
│  └──────────────────────────────────────┘                              │
│                          │                                              │
│                          ▼                                              │
│  ┌──────────────────────────────────────┐                              │
│  │  Statistical Comparison              │                              │
│  │  • Relative/absolute error           │                              │
│  │  • Order of magnitude check          │                              │
│  └──────────────────────────────────────┘                              │
│                          │                                              │
│                          ▼                                              │
│  ┌──────────────────────────────────────┐                              │
│  │  LLM Judge                           │                              │
│  │  • Consider all evidence             │                              │
│  │  • Make nuanced decision             │                              │
│  │  • Provide reasoning                 │                              │
│  └──────────────────────────────────────┘                              │
│                          │                                              │
│                          ▼                                              │
│  ✅ ACCEPT  |  ⚠️ MARGINAL  |  ❌ REJECT                                │
│  + Score + Reasoning + Recommendations                                  │
└─────────────────────────────────────────────────────────────────────────┘
```

### When to Use LLM-as-Judge:

| Use Case | Recommendation |
|----------|---------------|
| High-stakes decisions | ✅ LLM-as-Judge |
| Need explainable results | ✅ LLM-as-Judge |
| Complex semantic matching | ✅ LLM-as-Judge |
| High-volume screening | ⚠️ Use faster methods first |
| Real-time evaluation | ⚠️ Consider latency |
| Cost-sensitive | ⚠️ Consider alternatives |


In [ ]:
class LLMNumberMatcher:
    """
    Uses an LLM to match numbers between model answer and ground truth
    based on semantic meaning.
    """
    
    MATCHING_PROMPT = """You are an expert at matching numerical values between two texts based on their semantic meaning.

Given numbers extracted from a MODEL ANSWER and a GROUND TRUTH, match each number from the model answer to its corresponding number in the ground truth.

MODEL ANSWER NUMBERS:
{model_numbers}

GROUND TRUTH NUMBERS:
{truth_numbers}

Match numbers based on:
1. Semantic meaning (e.g., "revenue" should match "sales", "profit margin" should match "margin")
2. Context similarity
3. Unit compatibility

For each match:
- Indicate confidence level (high/medium/low)
- Explain why the numbers were matched

Also identify any unmatched numbers in either set."""

    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model
    
    def _format_numbers(self, numbers: List[ExtractedNumber]) -> str:
        """Format numbers for the prompt."""
        lines = []
        for i, num in enumerate(numbers, 1):
            lines.append(f"{i}. {num.semantic_label}: {num.value:,.4g} ('{num.original_text}')")
            lines.append(f"   Context: {num.context}")
        return "\n".join(lines)
    
    def match(
        self, 
        model_numbers: List[ExtractedNumber], 
        truth_numbers: List[ExtractedNumber]
    ) -> NumberMatchingResult:
        """Match numbers using LLM."""
        if not model_numbers or not truth_numbers:
            return NumberMatchingResult(
                matched_pairs=[],
                unmatched_model_numbers=model_numbers,
                unmatched_truth_numbers=truth_numbers,
                matching_notes="One or both sets are empty"
            )
        
        try:
            response = self.client.beta.chat.completions.parse(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an expert at semantic number matching. Return valid JSON."},
                    {"role": "user", "content": self.MATCHING_PROMPT.format(
                        model_numbers=self._format_numbers(model_numbers),
                        truth_numbers=self._format_numbers(truth_numbers)
                    )}
                ],
                response_format=NumberMatchingResult,
                temperature=0.0
            )
            return response.choices[0].message.parsed
        except Exception as e:
            print(f"Error in matching: {e}")
            return self._fallback_matching(model_numbers, truth_numbers)
    
    def _fallback_matching(
        self, 
        model_numbers: List[ExtractedNumber], 
        truth_numbers: List[ExtractedNumber]
    ) -> NumberMatchingResult:
        """Fallback matching by position and similar labels."""
        matched_pairs = []
        used_truth = set()
        unmatched_model = []
        
        for model_num in model_numbers:
            best_match = None
            best_score = 0
            
            for i, truth_num in enumerate(truth_numbers):
                if i in used_truth:
                    continue
                
                # Simple matching by label similarity
                score = 0
                model_label = model_num.semantic_label.lower()
                truth_label = truth_num.semantic_label.lower()
                
                # Check for common words
                model_words = set(model_label.split())
                truth_words = set(truth_label.split())
                common = model_words & truth_words
                
                if common:
                    score = len(common) / max(len(model_words), len(truth_words))
                
                # Same unit bonus
                if model_num.unit and model_num.unit == truth_num.unit:
                    score += 0.3
                
                if score > best_score:
                    best_score = score
                    best_match = (i, truth_num)
            
            if best_match and best_score > 0.2:
                used_truth.add(best_match[0])
                matched_pairs.append(NumberPair(
                    model_number=model_num,
                    truth_number=best_match[1],
                    match_confidence="medium" if best_score > 0.5 else "low",
                    match_reasoning=f"Matched by label similarity (score: {best_score:.2f})"
                ))
            else:
                unmatched_model.append(model_num)
        
        unmatched_truth = [t for i, t in enumerate(truth_numbers) if i not in used_truth]
        
        return NumberMatchingResult(
            matched_pairs=matched_pairs,
            unmatched_model_numbers=unmatched_model,
            unmatched_truth_numbers=unmatched_truth,
            matching_notes="Fallback label-similarity matching used"
        )


# Initialize matcher
matcher = LLMNumberMatcher(client, LLM_MODEL)

print("✅ LLMNumberMatcher initialized")
